# NB04 — Are the coordinates actually usable? Coordinate frames and the BGC queries

**Goal.** Move from *has coordinates* to *can be joined*. A table can carry contig, start, end
and strand and still be useless for collocation if nothing else is expressed in the same frame.
This notebook tests frame compatibility and then demonstrates the two queries the user named:

- **(a) genes near a BGC region** — a genomic interval join
- **(b) multi-domain search** — including the stricter *domain order* variant

**Discriminating query (run first).** The identifier-overlap check between
`kescience.mgnify.gene_bgc.contig_id` and `kescience.mgnify.gene.contig_id`. If that
intersection is empty, the coordinates cannot support collocation no matter how complete they
look, and that is the most consequential negative result the project could produce. Per
`RESEARCH_PLAN.md` it runs before anything confirmatory.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath("../src"))
import pandas as pd
from berdl_notebook_utils.setup_spark_session import get_spark_session
from census import CENSUS

spark = get_spark_session(app_name="acatlas_nb04")
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60); pd.set_option("display.max_rows", 120)
os.makedirs("../figures", exist_ok=True)
cen = pd.DataFrame(CENSUS)

## 1. Discriminating query — do BGC regions and genes share a contig namespace?

`gene_bgc` is partitioned by `biome_id`, so the cheap and honest test is to pick one biome,
pull its distinct contig ids from both tables, and intersect. Restricting to a shared biome
avoids a false negative that would arise purely from comparing disjoint slices.

In [2]:
biomes = spark.sql('''
SELECT biome_id, COUNT(*) AS n FROM kescience.mgnify.gene_bgc
GROUP BY biome_id ORDER BY n DESC LIMIT 5''').toPandas()
print("biomes with the most BGC rows:")
print(biomes.to_string(index=False))
BIOME = biomes.biome_id.iloc[0]
print(f"\nusing biome_id = {BIOME}")

biomes with the most BGC rows:
biome_id     n
    soil 35354

using biome_id = soil


In [3]:
ov = spark.sql(f'''
WITH b AS (SELECT DISTINCT contig_id FROM kescience.mgnify.gene_bgc WHERE biome_id = '{BIOME}'),
     g AS (SELECT DISTINCT contig_id FROM kescience.mgnify.gene     WHERE biome_id = '{BIOME}')
SELECT (SELECT COUNT(*) FROM b) AS bgc_contigs,
       (SELECT COUNT(*) FROM g) AS gene_contigs,
       (SELECT COUNT(*) FROM b JOIN g USING (contig_id)) AS shared_contigs
''').toPandas()
print(ov.to_string(index=False))
bc, gc, sc = int(ov.bgc_contigs[0]), int(ov.gene_contigs[0]), int(ov.shared_contigs[0])
print()
print(f"BGC contigs matched to a gene contig: {sc}/{bc} = {sc/bc:.1%}" if bc else "no BGC contigs")
print()
if sc == 0:
    print("RESULT: NO SHARED CONTIG NAMESPACE -- the coordinates cannot support collocation.")
else:
    print("RESULT: the namespaces DO overlap, so a genomic-interval join is possible with no")
    print("mapping table. This is what makes the MGnify family usable for 'genes near a BGC'.")

 bgc_contigs  gene_contigs  shared_contigs
       28494       3416781           28494

BGC contigs matched to a gene contig: 28494/28494 = 100.0%

RESULT: the namespaces DO overlap, so a genomic-interval join is possible with no
mapping table. This is what makes the MGnify family usable for 'genes near a BGC'.


In [4]:
# Sanity check that the ids are literally the same strings, not merely similar counts.
print("example shared contig ids:")
ex = spark.sql(f'''
WITH b AS (SELECT DISTINCT contig_id FROM kescience.mgnify.gene_bgc WHERE biome_id = '{BIOME}'),
     g AS (SELECT DISTINCT contig_id FROM kescience.mgnify.gene     WHERE biome_id = '{BIOME}')
SELECT contig_id FROM b JOIN g USING (contig_id) LIMIT 5''').toPandas()
print(ex.to_string(index=False))

example shared contig ids:


        contig_id
 MGYG000510928_26
 MGYG000510968_98
MGYG000510968_104
MGYG000510973_141
 MGYG000511051_68


## 2. Coordinate-frame map

Group the coordinate-bearing census tables by the frame their numbers are measured along. Two
tables can be joined on position **only inside one group**. This is the artifact most likely to
save someone a wasted afternoon: several of these frames look joinable and are not.

In [5]:
co = cen[cen.coords.isin(["yes", "packed"])][
    ["fq", "region_type", "coord_system", "coord_frame", "seq_ref", "strand"]].copy()
co["frame_group"] = co.coord_frame.fillna("UNKNOWN").map(lambda s: (
    "MGnify contig"            if "MGnify contig" in s else
    "pangenome cluster rep"    if "cluster representative" in s else
    "UniProt protein"          if "UniProt" in s else
    "PDB / structure"          if "PDB" in s else
    "own genome/contig (isolated)" if ("contig" in s or "chromosome" in s or "scaffold" in s) else
    "HMM / reference protein"  if ("HMM" in s or "reference protein" in s or "two frames" in s) else
    "UNRESOLVED (no sequence column)" if "UNKNOWN" in s else "other"))
print(co.groupby("frame_group").size().sort_values(ascending=False).to_string())
print()
for grp, sub in co.groupby("frame_group"):
    print(f"=== {grp} ({len(sub)} tables) ===")
    print(sub[["fq", "region_type", "coord_system", "strand"]].to_string(index=False))
    print()

frame_group
own genome/contig (isolated)    10
MGnify contig                    5
HMM / reference protein          3
UniProt protein                  2
other                            2
pangenome cluster rep            2

=== HMM / reference protein (3 tables) ===
                                       fq       region_type coord_system strand
         nmdc.results.pfam_annotation_gff    protein domain     multiple    NaN
   nmdc.results.annotation_kegg_orthology alignment segment     multiple    NaN
nmdc.results.annotation_enzyme_commission alignment segment     multiple    NaN

=== MGnify contig (5 tables) ===
                            fq               region_type coord_system      strand
     kescience.mgnify.gene_bgc biosynthetic gene cluster   nucleotide         NaN
  kescience.mgnify.gene_crispr              CRISPR array   nucleotide orientation
 kescience.mgnify.gene_defense            defense system   nucleotide         NaN
kescience.mgnify.gene_mobilome    mobile genetic ele

In [6]:
# Which frames contain BOTH a region-annotation table and a gene table? Only those can
# answer "genes near region X" without a mapping step.
REGIONISH = ["biosynthetic gene cluster", "CRISPR array", "defense system",
             "mobile genetic element", "protein domain/motif", "protein domain",
             "structural segment", "functional residue/site", "alignment segment"]
co["is_region"] = co.region_type.isin(REGIONISH)
co["is_gene"]   = co.region_type.fillna("").str.startswith("gene")
summ = co.groupby("frame_group")[["is_region", "is_gene"]].sum().astype(int)
summ["collocation_possible"] = (summ.is_region > 0) & (summ.is_gene > 0)
print(summ.to_string())
print()
ok = summ[summ.collocation_possible].index.tolist()
print("frames where a region-to-gene interval join needs NO mapping table:")
for f in ok: print("  -", f)
print()
print("frames with regions but no gene table in the same frame (mapping required):")
for f in summ[(summ.is_region > 0) & (summ.is_gene == 0)].index: print("  -", f)

                              is_region  is_gene  collocation_possible
frame_group                                                           
HMM / reference protein               3        0                 False
MGnify contig                         4        1                  True
UniProt protein                       2        0                 False
other                                 2        0                 False
own genome/contig (isolated)          1        8                  True
pangenome cluster rep                 2        0                 False

frames where a region-to-gene interval join needs NO mapping table:
  - MGnify contig
  - own genome/contig (isolated)

frames with regions but no gene table in the same frame (mapping required):
  - HMM / reference protein
  - UniProt protein
  - other
  - pangenome cluster rep


## 3. Query (a) — genes near a BGC region

The user's example. Executed on the MGnify frame, which section 1 showed is the only place this
is a single clean join. "Near" is parameterised by a flank so the query answers *inside or
within N kb of* a BGC, which is what BGC-boundary work actually needs.

In [7]:
FLANK = 5000
genes_near = spark.sql(f'''
SELECT b.bgc_id, b.nearest_mibig_class, b.contig_id,
       b.start_pos AS bgc_start, b.end_pos AS bgc_end,
       g.gene_id, g.start AS gene_start, g.end AS gene_end, g.strand,
       CASE WHEN g.start >= b.start_pos AND g.end <= b.end_pos THEN 'inside'
            ELSE 'flank' END AS placement
FROM kescience.mgnify.gene_bgc b
JOIN kescience.mgnify.gene g
  ON g.biome_id = b.biome_id AND g.contig_id = b.contig_id
 AND g.end   >= b.start_pos - {FLANK}
 AND g.start <= b.end_pos   + {FLANK}
WHERE b.biome_id = '{BIOME}'
ORDER BY b.bgc_id, g.start
''')
gn = genes_near.limit(400).toPandas()
print(f"returned {len(gn)} gene-BGC pairs (first 400) for biome {BIOME}")
print(gn.head(25).to_string(index=False))

returned 400 gene-BGC pairs (first 400) for biome soil
                     bgc_id nearest_mibig_class         contig_id  bgc_start  bgc_end             gene_id  gene_start  gene_end strand placement
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       8480    21413 MGYG000490806_00814        2775      3518      -     flank
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       8480    21413 MGYG000490806_00815        3515      3655      -     flank
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       8480    21413 MGYG000490806_00816        3688      4458      -     flank
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       8480    21413 MGYG000490806_00817        4448      5809      -     flank
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       8480    21413 MGYG000490806_00818        6276      6704      +     flank
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       848

In [8]:
# Scale of the result, and genes-per-BGC -- the number that tells you the join is sane.
stats = spark.sql(f'''
WITH j AS (
  SELECT b.bgc_id, g.gene_id,
         CASE WHEN g.start >= b.start_pos AND g.end <= b.end_pos THEN 1 ELSE 0 END AS inside
  FROM kescience.mgnify.gene_bgc b
  JOIN kescience.mgnify.gene g
    ON g.biome_id = b.biome_id AND g.contig_id = b.contig_id
   AND g.end >= b.start_pos - {FLANK} AND g.start <= b.end_pos + {FLANK}
  WHERE b.biome_id = '{BIOME}')
SELECT COUNT(*) AS gene_bgc_pairs, COUNT(DISTINCT bgc_id) AS bgcs_with_genes,
       SUM(inside) AS genes_inside,
       ROUND(COUNT(*) / COUNT(DISTINCT bgc_id), 1) AS genes_per_bgc,
       ROUND(SUM(inside) / COUNT(DISTINCT bgc_id), 1) AS genes_inside_per_bgc
FROM j''').toPandas()
print(stats.to_string(index=False))
print()
n_bgc = spark.sql(f"SELECT COUNT(*) c FROM kescience.mgnify.gene_bgc WHERE biome_id='{BIOME}'"
                  ).collect()[0][0]
print(f"BGCs in this biome: {n_bgc}; with >=1 gene found: {int(stats.bgcs_with_genes[0])} "
      f"({int(stats.bgcs_with_genes[0])/n_bgc:.0%})")
print()
print("A plausible genes-inside-per-BGC figure is the real check here: BGCs are typically a")
print("handful to a few dozen genes, so a number in that range means the interval join is")
print("operating in a genuinely shared coordinate frame, not just matching strings by luck.")

 gene_bgc_pairs  bgcs_with_genes  genes_inside  genes_per_bgc  genes_inside_per_bgc
         754313            35354        489978           21.3                  13.9

BGCs in this biome: 35354; with >=1 gene found: 35354 (100%)

A plausible genes-inside-per-BGC figure is the real check here: BGCs are typically a
handful to a few dozen genes, so a number in that range means the interval join is
operating in a genuinely shared coordinate frame, not just matching strings by luck.


In [9]:
gn.to_csv("../data/example_genes_near_bgc.csv", index=False)
print("wrote ../data/example_genes_near_bgc.csv")

# Add function to those genes -- the full use case is "which FUNCTIONS sit near a BGC".
func_near = spark.sql(f'''
SELECT b.nearest_mibig_class, e.cog_category, COUNT(*) AS n
FROM kescience.mgnify.gene_bgc b
JOIN kescience.mgnify.gene g
  ON g.biome_id = b.biome_id AND g.contig_id = b.contig_id
 AND g.end >= b.start_pos - {FLANK} AND g.start <= b.end_pos + {FLANK}
JOIN kescience.mgnify.gene_eggnog e
  ON e.gene_id = g.gene_id AND e.biome_id = g.biome_id
WHERE b.biome_id = '{BIOME}' AND e.cog_category IS NOT NULL AND e.cog_category <> '-'
GROUP BY b.nearest_mibig_class, e.cog_category
ORDER BY n DESC LIMIT 20''').toPandas()
print()
print("three-table join (BGC region -> gene coordinates -> eggNOG function):")
print(func_near.to_string(index=False))

wrote ../data/example_genes_near_bgc.csv



three-table join (BGC region -> gene coordinates -> eggNOG function):
nearest_mibig_class cog_category     n
         Saccharide            M 48873
         Saccharide            S 25588
         Polyketide            S 23844
              Other            S 18565
            Terpene            S 13905
              Other            M 13565
                NRP            S 13456
         Polyketide            I 12701
               RiPP            S 11803
         Polyketide           IQ  9702
         Polyketide            C  9359
                NRP            Q  9106
         Polyketide            Q  8571
            Terpene            I  8481
         Saccharide            E  8481
         Polyketide            K  7773
         Polyketide            E  7047
         Polyketide            M  7039
              Other            E  6984
         Saccharide            G  6521


## 4. Query (b) — multi-domain search

Two strengths of question, and BERDL supports them unequally:

1. **Co-occurrence** — "proteins carrying domains X *and* Y". Answerable from any
   coordinate-free table that lists domains per protein.
2. **Order and spacing** — "X then Y, N residues apart". Needs per-domain amino-acid
   coordinates, which only the span-bearing domain tables have.

Demonstrated on `kbase.ke_pangenome.interproscan_domains`, since it carries both Pfam
accessions and amino-acid spans on the same key.

In [10]:
# Co-occurrence: ABC transporter ATP-binding (PF00005) with a permease domain (PF00528).
PF_A, PF_B = "PF00005", "PF00528"
cooc = spark.sql(f'''
WITH d AS (
  SELECT gene_cluster_id, signature_acc, start, stop
  FROM kbase.ke_pangenome.interproscan_domains
  WHERE analysis = 'Pfam' AND signature_acc IN ('{PF_A}', '{PF_B}'))
SELECT COUNT(DISTINCT gene_cluster_id) AS proteins_with_both
FROM (SELECT gene_cluster_id FROM d GROUP BY gene_cluster_id
      HAVING COUNT(DISTINCT signature_acc) = 2)''').toPandas()
print(f"proteins carrying BOTH {PF_A} and {PF_B}: {int(cooc.proteins_with_both[0]):,}")

proteins carrying BOTH PF00005 and PF00528: 6,530


In [11]:
# Order + spacing: the query only coordinates make possible.
ordered = spark.sql(f'''
WITH a AS (SELECT gene_cluster_id, start AS a_start, stop AS a_stop, seq_len
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_A}'),
     b AS (SELECT gene_cluster_id, start AS b_start, stop AS b_stop
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_B}')
SELECT a.gene_cluster_id, a.seq_len, a.a_start, a.a_stop, b.b_start, b.b_stop,
       b.b_start - a.a_stop AS gap_aa,
       CASE WHEN a.a_stop < b.b_start THEN '{PF_A}->{PF_B}' ELSE '{PF_B}->{PF_A}' END AS domain_order
FROM a JOIN b USING (gene_cluster_id)
''')
# Aggregate over ALL pairs in Spark. A first version did ORDER BY gap_aa then LIMIT 300 and
# reported the median of that -- which is the extreme tail, not the distribution. Percentiles
# are computed over the full join; the CSV sample is taken WITHOUT an ordering.
agg = spark.sql(f'''
WITH a AS (SELECT gene_cluster_id, start AS a_start, stop AS a_stop
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_A}'),
     b AS (SELECT gene_cluster_id, start AS b_start, stop AS b_stop
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_B}'),
     j AS (SELECT a.gene_cluster_id, b.b_start - a.a_stop AS gap_aa,
                  CASE WHEN a.a_stop < b.b_start THEN 'A_then_B' ELSE 'B_then_A' END AS ord
           FROM a JOIN b USING (gene_cluster_id))
SELECT COUNT(*) AS pairs,
       COUNT(DISTINCT gene_cluster_id) AS proteins,
       SUM(CASE WHEN ord = 'A_then_B' THEN 1 ELSE 0 END) AS a_then_b,
       SUM(CASE WHEN ord = 'B_then_A' THEN 1 ELSE 0 END) AS b_then_a,
       percentile_approx(gap_aa, 0.25) AS p25,
       percentile_approx(gap_aa, 0.50) AS median_gap,
       percentile_approx(gap_aa, 0.75) AS p75
FROM j''').toPandas()
print(f"domain pairs over the FULL join ({PF_A} with {PF_B}):")
print(agg.to_string(index=False))
print()
ab, ba = int(agg.a_then_b[0]), int(agg.b_then_a[0])
print(f"{PF_A} before {PF_B}: {ab:,} ({ab/(ab+ba):.0%})")
print(f"{PF_B} before {PF_A}: {ba:,} ({ba/(ab+ba):.0%})")
print()
print("Reading this biologically: PF00528 is the ABC permease domain and PF00005 the")
print("ATP-binding domain. In fused transporters the permease is typically N-terminal and the")
print("ATPase C-terminal, so a majority of B-before-A is the expected architecture rather than")
print("an artifact -- which is itself a check that the amino-acid coordinates are meaningful.")

od = ordered.limit(300).toPandas()   # unordered sample for inspection
print()
print(f"inspection sample (unordered, {len(od)} rows):")
print(od.head(15).to_string(index=False))
od.to_csv("../data/example_multidomain_hits.csv", index=False)
print("wrote ../data/example_multidomain_hits.csv")

# PROTEIN-LEVEL order, deduplicated. A protein carrying repeat copies of either domain
# contributes several pairs to the figures above, so the pair-level percentage is
# pseudo-replicated -- the same class of error the schema_sig dedup in NB03 exists to avoid.
# One protein, one vote: a cluster counts as A_then_B only if the majority of its pairs are.
ordr = spark.sql(f'''
WITH a AS (SELECT gene_cluster_id, stop AS a_stop
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_A}'),
     b AS (SELECT gene_cluster_id, start AS b_start
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_B}'),
     j AS (SELECT a.gene_cluster_id,
                  CASE WHEN a.a_stop < b.b_start THEN 1 ELSE 0 END AS is_ab
           FROM a JOIN b USING (gene_cluster_id)),
     per_protein AS (SELECT gene_cluster_id, AVG(is_ab) AS frac_ab, COUNT(*) AS n_pairs
                     FROM j GROUP BY gene_cluster_id)
SELECT COUNT(*) AS proteins,
       SUM(CASE WHEN frac_ab > 0.5 THEN 1 ELSE 0 END) AS protein_a_then_b,
       SUM(CASE WHEN frac_ab < 0.5 THEN 1 ELSE 0 END) AS protein_b_then_a,
       SUM(CASE WHEN frac_ab = 0.5 THEN 1 ELSE 0 END) AS protein_tied,
       SUM(CASE WHEN n_pairs > 1 THEN 1 ELSE 0 END) AS proteins_with_repeats
FROM per_protein''').toPandas()
print()
print("PROTEIN-LEVEL (one cluster, one vote):")
print(ordr.to_string(index=False))
pab, pba = int(ordr.protein_a_then_b[0]), int(ordr.protein_b_then_a[0])
n_prot = int(ordr.proteins[0])
print()
print(f"  {PF_B} before {PF_A}: {pba:,}/{n_prot:,} proteins = {pba/n_prot:.0%}")
print(f"  proteins contributing >1 pair (repeat domains): "
      f"{int(ordr.proteins_with_repeats[0]):,} ({int(ordr.proteins_with_repeats[0])/n_prot:.0%})")
print()
print("The pair-level and protein-level percentages are reported separately; the protein-level")
print("figure is the one to quote, for the same reason NB03 quotes per-schema_sig rates.")

domain pairs over the FULL join (PF00005 with PF00528):
 pairs  proteins  a_then_b  b_then_a  p25  median_gap  p75
  6963      6530       385      6578 -412        -392 -371

PF00005 before PF00528: 385 (6%)
PF00528 before PF00005: 6,578 (94%)

Reading this biologically: PF00528 is the ABC permease domain and PF00005 the
ATP-binding domain. In fused transporters the permease is typically N-terminal and the
ATPase C-terminal, so a majority of B-before-A is the expected architecture rather than
an artifact -- which is itself a check that the amino-acid coordinates are meaningful.



inspection sample (unordered, 300 rows):
      gene_cluster_id  seq_len  a_start  a_stop  b_start  b_stop  gap_aa     domain_order
     BCAE01000031.1_9      652       31     178      466     649     288 PF00005->PF00528
CAJTIF010000001.1_388      472      302     422       87     255    -335 PF00528->PF00005
  CAJWNP010000048.1_3      621      372     530      107     286    -423 PF00528->PF00005
 CAKQQZ010000001.1_74      487      306     442       80     244    -362 PF00528->PF00005
 CAKQTN010000014.1_16     1053      822     972      585     771    -387 PF00528->PF00005
  CAKWAB010000011.1_5      671      371     520      108     286    -412 PF00528->PF00005
  CALTTZ010000004.1_5      608      289     419       59     255    -360 PF00528->PF00005
 CALVKZ010000095.1_11     1137      796     944      564     736    -380 PF00528->PF00005
     CSLW01000253.1_4      921      581     723       78     265    -645 PF00528->PF00005
     CSLW01000253.1_4      921      581     723      363  


PROTEIN-LEVEL (one cluster, one vote):
 proteins  protein_a_then_b  protein_b_then_a  protein_tied  proteins_with_repeats
     6530               350              6167            13                    395

  PF00528 before PF00005: 6,167/6,530 proteins = 94%
  proteins contributing >1 pair (repeat domains): 395 (6%)

The pair-level and protein-level percentages are reported separately; the protein-level
figure is the one to quote, for the same reason NB03 quotes per-schema_sig rates.


In [12]:
# Why this query is impossible on the coordinate-free tables, stated concretely.
pf_str = spark.sql('''
SELECT query_name, PFAMs FROM kbase.ke_pangenome.eggnog_mapper_annotations
WHERE PFAMs LIKE '%ABC_tran%' AND PFAMs LIKE '%,%' LIMIT 5''').toPandas()
print("the same information in eggnog_mapper_annotations.PFAMs:")
print(pf_str.to_string(index=False))
print()
print("Domain NAMES in a comma-joined string: co-occurrence is answerable with LIKE, order and")
print("spacing are not recoverable at all. That is the practical difference between the two")
print("kinds of multi-domain search, and it is exactly what the coordinate columns buy.")

the same information in eggnog_mapper_annotations.PFAMs:
           query_name                 PFAMs
  NZ_PELO01000314.1_1    ABC_tran,oligo_HPY
  JAGNPJ010000240.1_1      ABC_tran,DUF4162
    SAKH01000007.1_59          ABC_tran,CBS
      LR994474.1_2084    ABC_tran,oligo_HPY
NZ_MPSP01000001.1_655 ABC_membrane,ABC_tran

Domain NAMES in a comma-joined string: co-occurrence is answerable with LIKE, order and
spacing are not recoverable at all. That is the practical difference between the two
kinds of multi-domain search, and it is exactly what the coordinate columns buy.


## 5. Joinability verdict per region table

In [13]:
JOIN = [
 ("kescience.mgnify.gene_bgc", "BGC", "MGnify contig", "kescience.mgnify.gene",
  "direct", "same contig_id and biome_id; demonstrated in section 3"),
 ("kescience.mgnify.gene_crispr", "CRISPR array", "MGnify contig", "kescience.mgnify.gene",
  "direct", "same contig_id; strand column is named `orientation`"),
 ("kescience.mgnify.gene_defense", "defense system", "MGnify contig", "kescience.mgnify.gene",
  "direct", "also carries gene_id, so the linkage is pre-computed"),
 ("kescience.mgnify.gene_mobilome", "mobile element", "MGnify contig", "kescience.mgnify.gene",
  "direct", "same contig_id; expect -1 sentinels in start_pos"),
 ("refdata.smag.bgc", "BGC", "SMAG MAG/accession", "(none in this frame)",
  "blocked", "SMAG has no gene-coordinate table, so genes cannot be placed against these "
             "BGC intervals inside BERDL; carries BiG-SCAPE families the MGnify table lacks"),
 ("refdata.microagrobiome.bgc_catalog", "BGC", "none", "(n/a)",
  "impossible", "no sequence coordinates at all; lat/long only"),
 ("refdata.bvbrc.bvbrc_feature_fact", "gene/CDS", "BV-BRC contig", "itself",
  "self-contained but costly", "contig+start+end+strand+EC/KO in ONE table, so collocation is "
             "a self-join -- but coordinates are strings and identifiers are JSON"),
 ("refdata.jgi_virus.metavr_protein", "gene/CDS + Pfam", "IMG/VR scaffold", "itself",
  "self-contained", "scaffold_oid+start+end+strand+pfam array in one table; explode() the array"),
 ("refdata.jgi_mycocosm.features", "gene/CDS", "Mycocosm contig", "itself",
  "self-contained with a catch", "no `end` column -- derive start+length_bp"),
 ("refdata.fungal_db.pombase_gene_coords", "gene/exon/CDS", "S. pombe chromosome", "sibling tables",
  "direct", "gene/cds/exon hierarchy in one frame; function needs sgd/pombase xrefs"),
 ("kbase.ke_pangenome.interproscan_domains", "protein domain", "cluster representative protein",
  "itself", "protein-space only", "multi-domain order queries work (section 4); CANNOT reach a "
             "genome contig without a cluster-member mapping"),
 ("kbase.ke_pangenome.bakta_pfam_domains", "protein domain", "cluster representative protein",
  "itself", "protein-space only", "same frame as interproscan_domains"),
 ("kescience.interpro.protein2ipr", "protein domain", "UniProt protein", "itself",
  "protein-space only", "joins to any UniProt-keyed table; never directly to a genome"),
 ("nmdc.results.pfam_annotation_gff", "protein domain", "gene's protein + HMM model", "itself",
  "protein-space only", "despite the _gff name; no contig column anywhere in nmdc.results"),
 ("nmdc.results.annotation_kegg_orthology", "alignment", "query protein + KO reference", "itself",
  "protein-space only", "alignment extents, not domain boundaries"),
 ("kescience.fitnessbrowser.gene", "gene/CDS", "Fitness Browser scaffold", "genedomain",
  "cross-frame", "gene is nucleotide on scaffoldId, genedomain is amino-acid on locusId -- "
             "same database, two frames, joinable by key but NOT by position"),
 ("kbase.genomes.feature", "gene/CDS", "contig (via contig_x_feature bridge)", "itself",
  "bridged (1:1)", "CORRECTED from 'blocked': kbase.genomes.contig_x_feature maps all "
             "1,011,650,903 features onto 32,868,698 contigs with an exact 1:1 mapping, so one join "
             "places every feature and a self-join returns 14-15 CDS within +/-5 kb. Joinability is "
             "a property of the database, not of a single table's column list"),
 ("kbase.content.c_<genome>_features", "gene/CDS", "contig inside packed JSON", "sibling _cdss",
  "needs parsing", "location = [[contig,start,strand,length]]; parse before any interval logic"),
]
jd = pd.DataFrame(JOIN, columns=["region_table", "region_type", "frame", "gene_partner",
                                 "collocation", "note"])
print(jd.collocation.value_counts().to_string())
print()
print(jd.to_string(index=False))
jd.to_csv("../data/bgc_joinability.csv", index=False)

collocation
direct                         5
protein-space only             5
blocked                        1
impossible                     1
self-contained but costly      1
self-contained                 1
self-contained with a catch    1
cross-frame                    1
bridged (1:1)                  1
needs parsing                  1

                           region_table     region_type                                frame          gene_partner                 collocation                                                                                                                                                                                                                                                                                                          note
              kescience.mgnify.gene_bgc             BGC                        MGnify contig kescience.mgnify.gene                      direct                                                                      

In [14]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(13, 7.5))
order = ["direct", "bridged (1:1)", "self-contained", "self-contained with a catch",
         "self-contained but costly", "needs parsing",
         "protein-space only", "cross-frame", "blocked", "impossible"]
colour = {"direct": "#55a868", "bridged (1:1)": "#4f9d6b", "self-contained": "#6aa84f",
          "self-contained with a catch": "#b7c96b",
          "self-contained but costly": "#d4b85a", "needs parsing": "#dd8452",
          "protein-space only": "#4c72b0", "cross-frame": "#937860",
          "blocked": "#c44e52", "impossible": "#8b1a1a"}
missing = set(jd.collocation) - set(colour)
assert not missing, f"no colour for {missing}"
jd2 = jd.copy()
jd2["rank"] = jd2.collocation.map({k: i for i, k in enumerate(order)})
jd2 = jd2.sort_values(["rank", "region_table"], ascending=[True, False])
y = range(len(jd2))
ax.barh(list(y), [1] * len(jd2), color=[colour[c] for c in jd2.collocation])
ax.set_yticks(list(y))
ax.set_yticklabels([f"{r.region_table}  [{r.frame}]" for r in jd2.itertuples()], fontsize=8)
ax.set_xticks([])
ax.set_xlim(0, 1)
for i, r in enumerate(jd2.itertuples()):
    ax.text(0.015, i, r.collocation, va="center", ha="left", fontsize=8,
            color="white", fontweight="bold")
# Snapshot date read from the run, never hardcoded -- a baked-in date is the same
# stale-snapshot bug this project hit three times elsewhere.
_snap = json.load(open("../data/nb01_summary.json"))["semcat_generated_at"][:10]
ax.set_title("Can this region annotation be joined to genes by position?\n"
             f"BERDL coordinate frames, {_snap} (SemCat "
             f"{json.load(open('../data/nb01_summary.json'))['semcat_code_version']})",
             fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../figures/coordinate_frame_map.png", dpi=150, bbox_inches="tight")
print("wrote ../figures/coordinate_frame_map.png")
plt.show()

wrote ../figures/coordinate_frame_map.png


In [15]:
summary = dict(
    biome_tested=str(BIOME),
    bgc_contigs=bc, gene_contigs=gc, shared_contigs=sc,
    shared_contig_frac=float(sc / bc) if bc else None,
    gene_bgc_pairs=int(stats.gene_bgc_pairs[0]),
    bgcs_with_genes=int(stats.bgcs_with_genes[0]),
    bgcs_in_biome=int(n_bgc),
    genes_inside_per_bgc=float(stats.genes_inside_per_bgc[0]),
    proteins_with_both_pfams=int(cooc.proteins_with_both[0]),
    multidomain_pairs_total=int(agg.pairs[0]),
    multidomain_a_then_b=int(agg.a_then_b[0]),
    multidomain_b_then_a=int(agg.b_then_a[0]),
    multidomain_proteins=int(ordr.proteins[0]),
    multidomain_protein_a_then_b=int(ordr.protein_a_then_b[0]),
    multidomain_protein_b_then_a=int(ordr.protein_b_then_a[0]),
    multidomain_proteins_with_repeats=int(ordr.proteins_with_repeats[0]),
    median_interdomain_gap_aa=float(agg.median_gap[0]),
    interdomain_gap_p25=float(agg.p25[0]), interdomain_gap_p75=float(agg.p75[0]),
    collocation_direct=int((jd.collocation == "direct").sum()),
    collocation_bridged=int((jd.collocation == "bridged (1:1)").sum()),
    collocation_positionally_joinable=int(jd.collocation.isin(["direct", "bridged (1:1)"]).sum()),
    collocation_protein_only=int((jd.collocation == "protein-space only").sum()),
    collocation_blocked=int(jd.collocation.isin(["blocked", "impossible"]).sum()),
)
bad = {k: type(v).__name__ for k, v in summary.items()
       if not isinstance(v, (int, float, str, bool, type(None)))}
if bad: print("NON-SERIALIZABLE:", bad)
json.dump(summary, open("../data/nb04_summary.json", "w"), indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))

{
  "biome_tested": "soil",
  "bgc_contigs": 28494,
  "gene_contigs": 3416781,
  "shared_contigs": 28494,
  "shared_contig_frac": 1.0,
  "gene_bgc_pairs": 754313,
  "bgcs_with_genes": 35354,
  "bgcs_in_biome": 35354,
  "genes_inside_per_bgc": 13.9,
  "proteins_with_both_pfams": 6530,
  "multidomain_pairs_total": 6963,
  "multidomain_a_then_b": 385,
  "multidomain_b_then_a": 6578,
  "multidomain_proteins": 6530,
  "multidomain_protein_a_then_b": 350,
  "multidomain_protein_b_then_a": 6167,
  "multidomain_proteins_with_repeats": 395,
  "median_interdomain_gap_aa": -392.0,
  "interdomain_gap_p25": -412.0,
  "interdomain_gap_p75": -371.0,
  "collocation_direct": 5,
  "collocation_bridged": 1,
  "collocation_positionally_joinable": 6,
  "collocation_protein_only": 5,
  "collocation_blocked": 2
}
